#### Environment Check


In [1]:
import sys
print(sys.executable)


/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.venv/bin/python


#### Setup


In [2]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


#### Project Paths


In [3]:
cwd = Path.cwd()

if (cwd / "code").exists() and (cwd / "data").exists():
    LAB_DIR = cwd
elif (cwd.parent / "code").exists() and (cwd.parent / "data").exists():
    LAB_DIR = cwd.parent
else:
    LAB_DIR = Path("..").resolve()

CODE_DIR = LAB_DIR / "code"
DATA_DIR = LAB_DIR / "data"
REPORTS_DIR = LAB_DIR / "reports"

DATA_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

str(LAB_DIR)


'/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab'

#### Import Course Helpers


In [4]:
import sys

sys.path.append(str(CODE_DIR))

from ingest import load_faq_data, build_index
from evaluation_utils import RAGWithUsage, calc_total_price, map_progress


#### Load OpenAI Client


In [7]:
PROJECT_ROOT = LAB_DIR.parents[1]
ENV_PATH = PROJECT_ROOT / ".env"

loaded = load_dotenv(ENV_PATH)

print("Env file:", ENV_PATH)
print("Env loaded:", loaded)

openai_client = OpenAI()
MODEL = "gpt-5.4-mini"

Env file: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.env
Env loaded: True


#### Load Ground Truth


In [8]:
ground_truth_path = DATA_DIR / "ground_truth-new.csv"

df_ground_truth = pd.read_csv(ground_truth_path)
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth)


565

#### Load FAQ Documents


In [9]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm

len(documents)


113

#### Build Search Index


In [10]:
index = build_index(documents)

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

len(doc_idx)


113

#### Create RAG Assistant


In [11]:
assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    model=MODEL,
)


#### Test One RAG Answer


In [12]:
rec = ground_truth[0]
question = rec["question"]

answer_llm, usage = assistant.rag(question)

print(question)
print()
print(answer_llm)


What is the recommended way to begin the course and organize my study each week?

A good way to begin is to start with these resources:

1. The **LLM Zoomcamp docs**
2. The **general Zoomcamp logistics docs**
3. The **LLM Zoomcamp GitHub repository**

For weekly study, the typical workflow is:

1. **Watch the lesson videos**
2. **Work through the lesson notebooks/code**
3. **Read the homework instructions on GitHub**
4. **Submit your answers on the course platform before the deadline**

You can start anytime, and the materials and deadlines are available on the course platform.


#### Compare With Original Answer


In [13]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

print(answer_orig)


Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).

You can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).

A typical workflow is:

1. Watch the lesson videos.
2. Work through the lesson notebooks/code.
3. Read the homework instructions on GitHub.
4. Submit answers through the course platform before the deadline.

Homework is similar to the lesson flow, but uses a different dataset or slightly different task.


#### Create RAG Answer Function


In [14]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm, usage = assistant.rag(question)

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": original_doc["answer"],
        "document": doc_id,
    }

    return result, usage


#### Test Function


In [15]:
answer_record, usage = generate_rag_answer(ground_truth[0])

answer_record


{'question': 'What is the recommended way to begin the course and organize my study each week?',
 'answer_llm': 'Start with these resources:\n\n1. The **LLM Zoomcamp docs**\n2. The **general Zoomcamp logistics docs**\n3. The **LLM Zoomcamp GitHub repository**\n\nThen follow the usual weekly workflow:\n\n1. Watch the lesson videos\n2. Work through the lesson notebooks/code\n3. Read the homework instructions on GitHub\n4. Submit your answers on the course platform before the deadline\n\nYou can start whenever you want, since the videos and materials are available and deadlines are listed on the course management platform.',
 'answer_orig': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available

#### Generate RAG Answers


In [16]:
questions_to_process = ground_truth
# questions_to_process = ground_truth[:20]

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, questions_to_process, generate_rag_answer)


  0%|          | 0/565 [00:00<?, ?it/s]

#### Split Answers And Usage


In [21]:
answers = []
usages = []

for answer_record, usage in results:
    answers.append(answer_record)
    usages.append(usage)

len(answers)


565

#### Calculate RAG Cost


In [22]:
total_cost = calc_total_price(usages)

total_cost


0.10692014999999998

#### Save RAG Answers


In [23]:
df_answers = pd.DataFrame(answers)

output_path = DATA_DIR / "rag-answers-new.csv"
df_answers.to_csv(output_path, index=False)

output_path


PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data/rag-answers-new.csv')

#### Save RAG Report


In [24]:
report_path = REPORTS_DIR / "rag_evaluation.md"

report_lines = [
    "# RAG Answer Generation",
    "",
    f"- Questions processed: {len(df_answers)}",
    f"- Total cost: {total_cost}",
    f"- Output file: {output_path.name}",
]

with open(report_path, "w") as f:
    f.write("\n".join(report_lines))
    f.write("\n")

report_path


PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/reports/rag_evaluation.md')